In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightkurve as lk

# from temp_test_k2sc import k2sc_lc

import glob 
from tqdm import tqdm
from copy import deepcopy

from astropy.stats import sigma_clipped_stats

from sklearn.cluster import DBSCAN
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.gaussian_process import GaussianProcessRegressor

from Kakapo.photometry import forced_photometry
# from Kakapo.difference_image import create_diff_image
from Kakapo.difference_image import Difference_Imaging
from Kakapo.cleaning_curve import wavelet_denoise, correction_smoothing_lightcurve, flatten_and_clip_outliers, gauss_smooth, binned_averages

from hidden_prints.hidden_prints import HiddenPrints

%matplotlib widget

In [ ]:
def get_valid_baseline_indices(lc, frame_start, frame_end, base_range, min_points=42):
    """
    Expands the baseline region until at least `min_points` non-NaN values are found.
    
    Parameters:
        lc : array-like
            The lightcurve.
        frame_start : int
            Start of the event window.
        frame_end : int
            End of the event window.
        base_range : int
            Initial extension from event to define baseline.
        min_points : int
            Minimum number of non-NaN values required in the baseline.
    
    Returns:
        baseline_start : int
            Final start index of baseline region.
        baseline_end : int
            Final end index of baseline region.
        valid_inds : np.ndarray (bool)
            Boolean array marking valid baseline indices (non-NaN).
    """
    baseline_start = frame_start - base_range
    baseline_end = frame_end + base_range

    max_len = len(lc)
    frames = np.arange(max_len)

    while True:
        baseline_start = max(0, baseline_start)
        baseline_end = min(max_len, baseline_end)

        baseline_mask = ((frames > baseline_start) & (frames < frame_start)) | \
                        ((frames < baseline_end) & (frames > frame_end))
        valid_inds = baseline_mask & ~np.isnan(lc)

        if np.sum(valid_inds) >= min_points or (baseline_start == 0 and baseline_end == max_len):
            break

        baseline_start -= 1
        baseline_end += 1

    return baseline_start, baseline_end, baseline_mask

def _check_lc_significance(time, diff, start, end, x, y, flux_sign, distance = None, buffer = 1.1, base_range=2.85, grad_val = -60):
    cadence = time[1] - time[0]
    # cadence = cadence.value
    
    buffer = int(buffer/cadence)
    base_range = int(base_range/cadence)
    
    bjds = deepcopy(time)
    
    og_ind = np.arange(0, len(bjds))
    
    lc = forced_photometry(diff, x, y, None, bkg = True)
    
    mask = np.isfinite(lc) & np.isfinite(bjds) #& np.isfinite(flux_err)
    ind_mask = np.where(mask)[0]
    
    lc = lc[mask]
    bjds = bjds[mask]
    # flux_err = flux_err[mask]
    
    if distance is not None:
        distance = distance[mask]
        lc = gauss_smooth(bjds, lc, distance)
    else:
        lc = gauss_smooth(bjds, lc)
    
    sig_max, sig_med, lc_sig, indices = _significance_runner(lc, start, end, buffer, base_range, 
                                                                    min_points = 48, flux_sign = flux_sign, 
                                                                    grad_val = grad_val, binned = False)

    # merged = np.union1d(indices, mask) # Combine, remove duplicates, and sort
    
    return sig_max, sig_med, lc_sig, ind_mask, bjds, lc, og_ind

def _significance_runner(lc, start, end, buffer, base_range, 
                            min_points = 48, flux_sign = 1, grad_val = -60, binned = False):
    
    gradients = np.gradient(lc)
    
    frame_start = start - buffer
    frame_end = end + buffer
    if frame_start < 0:
        frame_start = 0
        frame_end += buffer
    if frame_end > len(lc):
        frame_end = len(lc) - 1 
        frame_start -= buffer
    
    if (frame_start < 0):
        frame_start = 0
    if (frame_end > len(lc)):
        frame_end = len(lc) - 1 
    
    baseline_start = frame_start - base_range
    baseline_end = frame_end + base_range
    if baseline_start < 0:
        baseline_start = 0
    if baseline_end > len(lc):
        baseline_end = len(lc) - 1
    
    frames = np.arange(len(lc))
    
    baseline_start, baseline_end, ind = get_valid_baseline_indices(lc, frame_start, frame_end, 
                                                                    base_range, min_points=min_points)
    
    print(baseline_start, baseline_end)
    
    med = np.nanmedian(lc[ind])
    gradmed = np.nanmedian(gradients[ind])
    std = np.nanstd(lc[ind], ddof = 1)
    gradstd = np.nanstd(gradients[ind], ddof = 1)
    lcevent = lc[int(start):int(end)]
    gradevent = gradients[int(start):int(end)]
    
    lc_sig = (lcevent - med) / std
    grad_sig = (gradients - gradmed) / gradstd
    
    indices = np.where((np.abs(grad_sig) < 10) & (gradients > grad_val))[0]
    
    try:
        sig_max = abs(np.nanmax(lc_sig))
        sig_med = abs(np.nanpercentile(lc_sig, 84))
    except:
        sig_med = -1
        sig_max = -1
    
    # print('Frame Start:', frame_start, frame_end, type(frame_start), type(frame_end))
    
    if np.nansum(np.isfinite(lc[int(frame_start):int(frame_end)])) < 5:
        sig_med = -1
    
    lc_sig = (lc - med) / std
    
    # if binned:
    #     return sig_max, sig_med
    # else: 
    return sig_max, sig_med, lc_sig * flux_sign, indices

In [ ]:
# def detected_events(time, diff, events, siglim = 2):
    
#     cluster_ids = np.unique(events['cluster'])
    
#     full_events = pd.DataFrame(columns=['cluster', 'sig_max', 'sig_84', 
#                                         'frame_min', 'frame_max', 
#                                         'bjds', 'flux', 'flux_err'])
    
#     new_stars = pd.DataFrame(columns=events.columns)
    
#     events['lc_sig'] = -1*np.ones(len(events))
    
#     # for cluster_id in tqdm(cluster_ids, desc='Clusters'):
#     for cluster_id in cluster_ids:
#         # print(cluster_id, self.campaign, self.targetid)
#         cluster = events[events['cluster'] == cluster_id]
        
#         if len(cluster) < 5:
#             continue
        
#         frame_min = int(cluster['frame'].min())
#         frame_max = int(cluster['frame'].max())
        
#         time_min = time[frame_min]
#         time_max = time[frame_max]
        
#         x, _, xstd = sigma_clipped_stats(cluster['xcentroid'].values, sigma = 3)
#         y, _, ystd = sigma_clipped_stats(cluster['ycentroid'].values, sigma = 3)
        
#         if (xstd >= 0.9) | (ystd >= 0.9):
#             continue
        
#         args = _check_lc_significance(time, diff, frame_min, frame_max, time_min, time_max, x, y, 1, 
#                                             buffer = 1.2, base_range=2.6, grad_val = -60)
        
#         sig_max, sig_med, lc_sig, indices, bjds, flux, flux_err = args
        
#         print(sig_max, sig_med)
        
#         if sig_med < siglim:
#             continue
        
#         filtered_cluster = cluster[cluster['frame'].isin(indices)]
#         filtered_cluster = filtered_cluster.reset_index(drop=True)
#         frame_inds = filtered_cluster['frame'].values.astype(int)
        
#         filtered_lc_sig = lc_sig[frame_inds]
#         filtered_lc_sig_indices = filtered_lc_sig > siglim
        
#         filtered_frame_values = filtered_cluster['frame'].values[filtered_lc_sig_indices]
#         filtered_frame_values = np.sort(filtered_frame_values, axis=None) 
        
#         filtered_lc_sig = filtered_lc_sig[filtered_lc_sig_indices]
#         filtered_final_cluster = filtered_cluster[filtered_cluster['frame'].isin(filtered_frame_values)]
#         filtered_final_cluster['lc_sig'] = filtered_lc_sig
        
#         if len(filtered_lc_sig) < 5:
#             continue
#         else:
#             # print(len(filtered_lc_sig))
#             filtered_cluster['cluster'] = len(full_events)
#             new_stars = pd.concat([new_stars, filtered_cluster])
#             full_events.loc[len(full_events)] = [len(full_events), sig_max, sig_med, 
#                                                     filtered_final_cluster['frame'].min(), 
#                                                     filtered_final_cluster['frame'].max(), 
#                                                     bjds, flux, flux_err]
    
#     new_stars = new_stars.reset_index(drop=True)
    
#     return full_events, new_stars
    

In [ ]:
def _tpf_addition(tpf_info, tpf_input):
    
    if tpf_input.campaign is None:
        campaign = tpf_input.quarter
        mission = 'Kepler'
        print(f"Adding TPF {tpf_input.targetid} from {mission} quarter {campaign}")
    else:
        campaign = tpf_input.campaign
        mission = 'K2'
    
    tpf_info.loc[len(tpf_info)] = [mission, campaign, tpf_input.targetid, tpf_input.ra, tpf_input.dec, 
                                   tpf_input.flux.value, tpf_input.flux_err.value, tpf_input.quality, 
                                   tpf_input.pos_corr1, tpf_input.pos_corr2, tpf_input.time, tpf_input.cadenceno, 
                                   tpf_input.hdu]
    
    return tpf_info

def _check_tpf_type(tpf_input):
    
    tpf_info = pd.DataFrame(columns=['mission', 'campaign', 'targetid', 'ra', 'dec', 'flux', 'flux_err', 
                                     'quality', 'pos_corr1', 'pos_corr2', 'time', 'cadenceno', 'hdu'])
        
    for i in range(len(tpf_input)):
        if isinstance(tpf_input[i], lk.targetpixelfile.KeplerTargetPixelFile):
            tpf_info = _tpf_addition(tpf_info, tpf_input[i])
        
    return tpf_info

In [ ]:
def _grouping(corr: pd.DataFrame, f_dist: int = 48) -> pd.DataFrame | None:
    """
    Group detections with DBSCAN in O(N log N) time, using only C/Fortran
    code paths from scikit-learn (no Python callback per point pair).
    """
    if corr.empty:
        return None

    # Scale the frame axis so that `eps` of 1.25 encloses ±f_dist frames.
    data = corr[['xcentroid', 'ycentroid', 'frame']].values.astype(np.float32)
    data[:, 2] *= 1.25 / f_dist

    db = DBSCAN(eps=1.25,
                min_samples=5,
                metric='euclidean',           # now fully compiled
                algorithm='auto',        # fastest for 3‑D Euclidean
                n_jobs=1)                     # keep it serial – you already parallelise at a higher level
    labels = db.fit_predict(data)

    corr = corr.assign(cluster=labels)
    corr = corr[corr.cluster != -1]           # drop noise points

    return corr if not corr.empty else None

def initial_filter(df):
    df = df[(df.fwhm > 0.9) & (df.snr >= 4) & 
            (df.psfdiff <= 1.5) & (df.poisson_thresh >= 1) & 
            (abs(df.correlation) >= 0.05)]
    
    return df

def mask_detections(correlation, psfdiff, fwhm, snr, 
                    roundness, poisson_thresh, xstd, ystd):
    
    mask =  (correlation >= 0.05) & (psfdiff <= 1.5) & \
            (fwhm <= 5) & (fwhm >= 0.95) & \
            (snr >= 4) & (snr < 10000) & (abs(roundness) <= 0.8) & \
            (poisson_thresh >= 1) & \
            (xstd <= 0.5) & (ystd <= 0.5)
    
    print(f'mask: {mask}')
    
    return ~mask

In [ ]:
def access_tpfs():
    """
    """

    test_case = []

    lightkurve_file_folder = '/Users/zgl12/.lightkurve/cache/mastDownload/K2/'

    files = sorted(glob.glob(lightkurve_file_folder + '*211394078*/*.fits.gz'))

    for file in tqdm(files, desc='Reading TPFs'):
        tpf = lk.read(file, quality_bitmask = 'none')
        test_case.append(tpf)
        
    return test_case

In [ ]:
tpf = access_tpfs()
tpf_info = _check_tpf_type(tpf)
tpf_info

In [ ]:
csv_file_1 = '/Users/zgl12/Modules/Kakapo/Data/csv_files/c5/c5_t211394078.csv'

In [ ]:
df_1 = pd.read_csv(csv_file_1)
# df_2 = pd.read_csv(csv_file_2)

df_1 = initial_filter(df_1)
# df_2 = initial_filter(df_2)

df_1 = _grouping(df_1, f_dist = 48)
# df_2 = _grouping(df_2, f_dist = 18)

In [ ]:
clusters_id = np.unique(df_1.cluster.values)

for cluster in clusters_id:
    temp_cluster = df_1[df_1.cluster == cluster]
    
    x, _, xstd = sigma_clipped_stats(temp_cluster.xcentroid.values, sigma = 3)
    y, _, ystd = sigma_clipped_stats(temp_cluster.ycentroid.values, sigma = 3)
    
    print(cluster, 'frame:', int(np.nanmin(temp_cluster.frame.values)), 'to', int(np.nanmax(temp_cluster.frame.values)))
    print(f'x = {x:.2f} +/- {xstd:.2f}')
    print(f'y = {y:.2f} +/- {ystd:.2f}')
    print()
    
    # break 

In [ ]:
temp_cluster

In [ ]:
temp_df_1 = df_1[df_1.cluster == 0]

In [ ]:
x, _, xstd = sigma_clipped_stats(temp_df_1.xcentroid.values, sigma = 3)
y, _, ystd = sigma_clipped_stats(temp_df_1.ycentroid.values, sigma = 3)
correlation, _, _ = sigma_clipped_stats(temp_df_1.correlation.values, sigma = 3)
psfdiff, _, _  = sigma_clipped_stats(temp_df_1.psfdiff.values, sigma = 3)
snr, _, _  = sigma_clipped_stats(temp_df_1.snr.values, sigma = 3)
fwhm, _, _  = sigma_clipped_stats(temp_df_1.fwhm.values, sigma = 3)
roundness, _, _  = sigma_clipped_stats(temp_df_1.roundness.values, sigma = 3)
poisson_thresh, _, _  = sigma_clipped_stats(temp_df_1.poisson_thresh.values, sigma = 3)

mask_detections(correlation, psfdiff, fwhm, snr, 
                    roundness, poisson_thresh, xstd, ystd)

# x_2, _, xstd = sigma_clipped_stats(temp_df_2.xcentroid.values, sigma = 3)
# y_2, _, ystd = sigma_clipped_stats(temp_df_2.ycentroid.values, sigma = 3)

In [ ]:
print(f'Corr: {correlation}')
print(f'PSF Diff.: {psfdiff}')
print(f'SNR: {snr}')
print(f'Round: {roundness}')

In [ ]:
epsf = np.genfromtxt('/Users/zgl12/Modules/Kakapo/epsf_data.txt')
epsf = epsf[2:-2,2:-2]

# tpf = tpf_info.iloc[0]

# # ref, diffs, poisson_noise, thrusters, distance = create_diff_image(tpf_info.iloc[0], epsf, tol=0.2)

chunky_bird = Difference_Imaging(tpf_info.iloc[0], epsf, tol=0.2)

ref = chunky_bird.ref
noise = chunky_bird.noise
diffs = chunky_bird.diffs
# thrusters = chunky_bird.thrusters
distance = chunky_bird.distance

# dxs = -chunky_bird.dxs
# dys = -chunky_bird.dys


In [ ]:
start = int(np.nanmin(temp_df_1.frame.values))
end = int(np.nanmax(temp_df_1.frame.values))
time = tpf_info.iloc[0].time.value

sig_max, sig_med, lc_sig, mask, bjds, lc = _check_lc_significance(time, diffs, start, end, x, y, 1, distance,
                                                                  buffer = 1.1, base_range=2.85, grad_val = -60)

In [ ]:
sig_max, sig_med

In [ ]:
# bjds = deepcopy(time)

# lc_og = forced_photometry(diffs, x, y, None, bkg = True)

# mask = np.isfinite(lc_og) & np.isfinite(bjds) #& np.isfinite(flux_err)

# lc_og = lc_og[mask]
# bjds = bjds[mask]
# # flux_err = flux_err[mask]

# lc = gauss_smooth(bjds, lc_og)

In [ ]:
tpf = tpf_info.iloc[0]

targetid = tpf.targetid
time = tpf.time.value

flux = forced_photometry(diffs, x, y, epsf, bkg = True, method = 'aperture')
# flux_err = forced_photometry(noise, x, y, epsf, bkg = True, method = 'aperture')
# xs = dxs
# ys = dys
# quality = tpf.quality
# campaign = tpf.campaign
# cadenceno = tpf.cadenceno
# hdu = tpf.hdu

mask = np.isfinite(flux) & np.isfinite(time) #& np.isfinite(flux_err)

# # print(np.sum(~mask))

flux = flux[mask]
# time = time[mask]
# # flux_err = flux_err[mask]
# # # xs = xs[mask]
# # # ys = ys[mask]
# # # cadenceno = cadenceno[mask]

# corr_flux = gauss_smooth(time, flux, n_samples = 1)

# # with HiddenPrints():
# data_su = k2sc_lc(targetid, time, flux, flux_err, xs, ys, quality, campaign, cadenceno, hdu)
# data = data_su.k2sc()


In [ ]:
fig = plt.figure(figsize=(12.0,8.0))
plt.plot(bjds, flux,'.',label = "Uncorrected")
plt.plot(bjds, lc,'.',label = "Corrected")
# plt.plot(bjds, lc_og,'.',label = "Uncorr")
# plt.axvline(time[706])
# detrended = lc.corr_flux-lc.tr_time + np.nanmedian(lc.tr_time)
# plt.plot(time, data,'.',label="K2SC")
plt.legend()
plt.xlabel('BJD')
plt.ylabel('Flux')
# plt.title('WASP-55',y=1.01)
plt.show()

In [ ]:
lc